# 임시 분석: WikiText-103 word 150토큰은 GPT-2 BPE 몇 토큰인가?

Colab의 CPU 런타임에서 위에서부터 실행합니다. 저장소 clone·모델 학습·GPU가 필요 없습니다.

- 데이터: `Salesforce/wikitext`, **`wikitext-103-v1`**. raw 버전이 아닙니다.
- word 기준: 원본 줄의 `line.split()` + 줄마다 EOS 1개. 빈 줄도 EOS 1개입니다.
- BPE 기준: `openai-community/gpt2`, 원본 공백·대소문자 보존, 줄 끝 CR/LF 제거, 줄마다 EOS 1개. prefix space·BOS·padding·truncation을 추가하지 않습니다.
- 이 처리는 현재 GRT 저장소의 word/BPE 준비 방식과 맞춥니다. 자연어 문장 길이 대신 **연속 스트림의 150 word-token 구간**에 대응하는 BPE 토큰 수를 측정합니다.

기본은 train 전체입니다. 빠르게 먼저 보려면 `SPLIT = 'validation'`으로 바꾸세요. 표본만 보려면 `MAX_ROWS`를 지정할 수 있지만, 그 결과는 전체 train 통계가 아닙니다.

In [ ]:
%pip -q install "datasets>=3,<4" "transformers==4.45.2" numpy tqdm

In [ ]:
from pathlib import Path
import io
import json
import re
import time

import numpy as np
from datasets import load_dataset
from transformers import GPT2TokenizerFast
from tqdm.auto import tqdm

DATASET_ID = 'Salesforce/wikitext'
DATASET_CONFIG = 'wikitext-103-v1'
DATASET_REVISION = 'main'
TOKENIZER_ID = 'openai-community/gpt2'
TOKENIZER_REVISION = 'main'
SPLIT = 'train'            # train / validation / test
MAX_ROWS = None           # None: 전체 split; 예: 20000은 앞쪽 20000행만 분석
WORD_SEGMENT_LEN = 150
BATCH_SIZE = 256
OUTPUT = Path('wikitext_word150_bpe_summary.json')

assert SPLIT in ('train', 'validation', 'test')
assert type(WORD_SEGMENT_LEN) is int and WORD_SEGMENT_LEN > 0
assert type(BATCH_SIZE) is int and BATCH_SIZE > 0
assert MAX_ROWS is None or (type(MAX_ROWS) is int and MAX_ROWS > 0)

각 원본 줄을 통째로 BPE 토큰화한 뒤 문자 offset으로 word 위치와 정렬합니다. 단어들을 공백으로 다시 연결해 토큰화하지 않으므로 원래 공백과 줄별 EOS가 유지됩니다.

공백만 나타내는 BPE 토큰은 뒤쪽 word에, 마지막 word 뒤의 공백은 그 줄의 EOS에 귀속합니다. 이 경계 규칙에 따라 연속 word 스트림을 150개씩 나누어 BPE 수를 합합니다. 마지막 150개 미만의 word 구간은 전체 토큰 비율에 포함하지만 완전한 구간의 길이 분포에서는 제외합니다.

In [ ]:
def source_lines(rows):
    for row in rows:
        text = row['text']
        payload = text if text.endswith('\n') else text + '\n'
        # 현재 downloader가 내보내는 줄과 같은 방식으로 읽습니다.
        with io.StringIO(payload, newline='') as stream:
            for line in stream:
                yield line.rstrip('\r\n')


def bpe_counts_per_word(text, encoded):
    word_ends = np.fromiter((m.end() for m in re.finditer(r'\S+', text)), dtype=np.int64)
    bpe_ends = np.fromiter((end for start, end in encoded.offsets), dtype=np.int64)
    indices = np.searchsorted(word_ends, bpe_ends, side='left')
    counts = np.bincount(indices, minlength=len(word_ends) + 1)
    counts[-1] += 1  # 원본 줄의 마지막 word 뒤에 추가하는 EOS
    return counts


def measure(rows, tokenizer, *, word_segment_len, batch_size):
    pending = np.empty(0, dtype=np.int64)
    windows = []
    totals = dict(source_lines=0, word_tokens=0, bpe_tokens=0)
    texts = []

    def consume(texts, pending):
        encodings = tokenizer.backend_tokenizer.encode_batch(texts, add_special_tokens=False)
        pieces = []
        for text, encoded in zip(texts, encodings):
            counts = bpe_counts_per_word(text, encoded)
            pieces.append(counts)
            totals['source_lines'] += 1
            totals['word_tokens'] += len(counts)
            totals['bpe_tokens'] += len(encoded.ids) + 1
        counts = np.concatenate([pending, *pieces])
        end = len(counts) // word_segment_len * word_segment_len
        if end:
            windows.append(counts[:end].reshape(-1, word_segment_len).sum(axis=1))
        return counts[end:]

    for text in source_lines(rows):
        texts.append(text)
        if len(texts) == batch_size:
            pending = consume(texts, pending)
            texts = []
    if texts:
        pending = consume(texts, pending)
    lengths = np.concatenate(windows) if windows else np.empty(0, dtype=np.int64)
    if not len(lengths):
        raise ValueError('완전한 word 구간이 없습니다. MAX_ROWS를 늘려주세요.')
    assert int(lengths.sum()) + int(pending.sum()) == totals['bpe_tokens']
    assert len(lengths) * word_segment_len + len(pending) == totals['word_tokens']
    ratio = totals['bpe_tokens'] / totals['word_tokens']
    return dict(
        **totals,
        word_segment_len=word_segment_len,
        full_word_segments=len(lengths),
        tail_word_tokens=len(pending),
        tail_bpe_tokens=int(pending.sum()),
        bpe_per_word_stream_ratio=ratio,
        equivalent_bpe_length_from_stream_ratio=word_segment_len * ratio,
        bpe_150_equivalent_word_tokens=150 / ratio,
        bpe_length_for_full_word_segments=dict(
            mean=float(lengths.mean()), std=float(lengths.std()),
            min=int(lengths.min()), p10=float(np.percentile(lengths, 10)),
            median=float(np.median(lengths)), p90=float(np.percentile(lengths, 90)),
            p95=float(np.percentile(lengths, 95)), max=int(lengths.max())),
    ), lengths

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained(
    TOKENIZER_ID, revision=TOKENIZER_REVISION, add_prefix_space=False)
assert len(tokenizer) == 50257 and tokenizer.eos_token_id == 50256
tokenizer.backend_tokenizer.no_truncation()
tokenizer.backend_tokenizer.no_padding()

data = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT,
                    revision=DATASET_REVISION, streaming=True)
rows = data if MAX_ROWS is None else data.take(MAX_ROWS)
started = time.perf_counter()
summary, lengths = measure(tqdm(rows, desc=f'{SPLIT} rows'), tokenizer,
                           word_segment_len=WORD_SEGMENT_LEN, batch_size=BATCH_SIZE)
summary.update(
    dataset=DATASET_ID, dataset_config=DATASET_CONFIG,
    dataset_revision=DATASET_REVISION, split=SPLIT, max_rows=MAX_ROWS,
    tokenizer=TOKENIZER_ID, tokenizer_revision=TOKENIZER_REVISION,
    tokenizer_commit=tokenizer.init_kwargs.get('_commit_hash'),
    elapsed_seconds=time.perf_counter() - started,
    convention='word: whitespace split + line EOS; BPE: original line + EOS; whitespace assigned to following word/EOS',
)
OUTPUT.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
stats = summary['bpe_length_for_full_word_segments']
print(f"word {WORD_SEGMENT_LEN}토큰 → BPE 평균 {stats['mean']:.1f}, 중앙값 {stats['median']:.1f}")
print(f"BPE 길이의 10~90% 분위수: {stats['p10']:.1f} ~ {stats['p90']:.1f}")
print(f"현재 BPE 150토큰 → 평균적으로 word 약 {summary['bpe_150_equivalent_word_tokens']:.1f}토큰 분량")
print(f"분석한 완전한 word 구간: {summary['full_word_segments']:,}개")
print('공유할 결과 파일:', OUTPUT.resolve())

마지막 셀의 숫자나 `wikitext_word150_bpe_summary.json`을 공유하면 됩니다. 전체 split 기준인지 표본 기준인지도 결과에 기록됩니다.

이 환산은 같은 텍스트 양을 비교하기 위한 값입니다. segment를 변경할 때 LR·학습량·BPTT를 자동 결정하거나, word-level PPL과 BPE-token PPL을 같은 척도로 환산하지 않습니다. 저장소의 학습 코드·config·정식 노트북은 변경하지 않습니다.